In [ ]:
import pandas as pd

# === FILE PATHS ===
mapping_file = "D:/Tushar/main_with_subs_only.xlsx"
indent_file  = "D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"

# === LOAD FILES ===
print("Loading mapping file...")
df_mapping = pd.read_excel(mapping_file)

print("Loading Monthly Indent (Sheet1)...")
df_indent = pd.read_excel(indent_file, sheet_name="Sheet1")

# === RENAME COLUMNS ===
# Mapping file
df_mapping = df_mapping.rename(columns={
    'Main_Label': 'Child_Part',
    'Sub_Label':  'Switch_Part',
    'Sub_Count':  'Qty_per_Switch'
})

# Monthly indent file
df_indent = df_indent.rename(columns={
    'Part number': 'Switch_Part'
})

# === MONTH COLUMNS ===
month_cols = ["Feb'26","Mar'26","Apr'26","May'26","Jun'26","Jul'26"]

# === MERGE ===
print("Merging data...")
df_merged = pd.merge(
    df_mapping[['Child_Part','Switch_Part','Qty_per_Switch']],
    df_indent[['Switch_Part'] + month_cols],
    on='Switch_Part',
    how='left'
)

print(f"Merged rows: {len(df_merged)}")

# === CLEAN MONTH NAMES ===
clean_months = [m.replace("'", "") for m in month_cols]

# === CALCULATE DAILY + 2 DAY REQUIREMENT ===
for month, clean in zip(month_cols, clean_months):

    daily_col = f"Daily_{clean}"
    two_col   = f"2Days_{clean}"

    # Daily switch demand
    df_merged[daily_col] = df_merged[month] / 30.0

    # Child requirement
    df_merged[two_col] = df_merged[daily_col] * df_merged['Qty_per_Switch'] * 2

# === TOTAL REQUIREMENT PER CHILD ===
print("Calculating totals per child...")

agg_dict = {f"Daily_{c}": 'sum' for c in clean_months}

totals = df_merged.groupby('Child_Part', as_index=False).agg(agg_dict)

for c in clean_months:
    totals[f"2Days_{c}"] = totals[f"Daily_{c}"] * 2

# Save totals
totals_file = "Child_Totals_2Days_Per_Month.xlsx"
totals.to_excel(totals_file, index=False)

print("Saved totals file:", totals_file)

# === DETAILED BREAKDOWN ===
detail_cols = (
    ['Child_Part','Switch_Part','Qty_per_Switch'] +
    month_cols +
    [f"Daily_{c}" for c in clean_months] +
    [f"2Days_{c}" for c in clean_months]
)

details = df_merged[detail_cols]

details_file = "Child_Detailed_Breakdown_2Days.xlsx"
details.to_excel(details_file, index=False)

print("Saved detailed file:", details_file)

print("\n✅ DONE — requirement calculated correctly.")
